In [0]:
source_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/pyspark_catalog/default/task_of_scd")

source_df.display()

In [0]:
updated_df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/Volumes/pyspark_catalog/default/target_scd")

updated_df.display()

In [0]:
source_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dim_customer")

In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "dim_customer")

deltaTable.alias("target").merge(
    updated_df.alias("source"),
    "target.customer_id = source.customer_id"
).whenMatchedUpdate(
    set={
        "customer_name": "source.customer_name",
        "city": "source.city",
        "email": "source.email",
        "department": "source.department",
        "salary": "source.salary",
        "last_updated": "source.last_updated"
    }
).whenNotMatchedInsert(
    values={
        "customer_id": "source.customer_id",
        "customer_name": "source.customer_name",
        "city": "source.city",
        "email": "source.email",
        "department": "source.department",
        "salary": "source.salary",
        "last_updated": "source.last_updated"
    }
).execute()

In [0]:
%sql
SELECT *
FROM dim_customer
ORDER BY customer_id;